## CI/CD support scripts

  * Enable automation of the model lifecycle.
  * Ensure consistency across environments (train/test/deploy).
  * Reduce human error and support repeatable, reliable deployments.

In [ ]:
# train_logistic.py

import pandas as pd
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# SageMaker input directory
INPUT_PATH = "/opt/ml/input/data/train/cardio_engineered.csv"
OUTPUT_PATH = "/opt/ml/model"

try:
    print("✅ Reading input CSV from:", INPUT_PATH)
    df = pd.read_csv(INPUT_PATH)

    # Encode categories
    df["age_group"] = df["age_group"].map({'30s':1, '40s':2, '50s':3, '60s':4, '70s':5})
    df["cholesterol_label"] = df["cholesterol_label"].map({'Normal':1, 'Above Normal':2, 'Well Above Normal':3})
    df["bp_category"] = df["bp_category"].map({'normal':1, 'stage1':2, 'stage2':3})
    df["bmi_category"] = df["bmi_category"].map({'normal':1, 'overweight':2, 'obese':3})
    df.dropna(inplace=True)

    X = df.drop("cardio", axis=1)
    y = df["cardio"]

    # Scale and split
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # Train model
    model = LogisticRegression(max_iter=2000)
    model.fit(X_train, y_train)

    # Save model
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    joblib.dump(model, os.path.join(OUTPUT_PATH, "model.joblib"))

    print("✅ Training completed and model saved.")
except Exception as e:
    print("❌ ERROR during training:", str(e))
    raise

## train_model.py
### Trains the logistic regression model using the preprocessed and feature-engineered dataset.
### Key functions:
  * Loads training data (e.g., CSV from S3 or local path).
  * Initializes a logistic regression model using sklearn.
  * Trains the model on the dataset.
  * Serializes and saves the trained model as a .joblib file for downstream packaging.

In [ ]:
#  package_model.py

import tarfile
import boto3
import os

MODEL_DIR = "model"
TAR_FILE = "logistic_model.tar.gz"
S3_BUCKET = "sagemaker-us-east-1-993768311527"
S3_KEY = f"logistic/{TAR_FILE}"

# Package model and inference script for Script Mode
with tarfile.open(TAR_FILE, "w:gz") as tar:
    tar.add(f"{MODEL_DIR}/logistic_model.pkl", arcname="logistic_model.pkl")
    tar.add(f"{MODEL_DIR}/inference.py", arcname="inference.py")

# Upload to S3
s3 = boto3.client("s3")
s3.upload_file(TAR_FILE, S3_BUCKET, S3_KEY)
print(f"Uploaded: s3://{S3_BUCKET}/{S3_KEY}")

## package_model.py
### Packages the trained model artifact and the inference script into a deployable format.

### Key functions:
  * Creates a .tar.gz archive using Python’s tarfile module.
  * Adds the trained model file (e.g., logistic_model.joblib) and inference logic (inference.py) to the archive.
  * Prepares the package for upload to S3 or SageMaker model registry.
  

In [ ]:
#  deploy_model.py

import boto3
import time

timestamp = int(time.time())

MODEL_NAME = f"logistic-scriptmode-{timestamp}"
TRANSFORM_JOB_NAME = f"logistic-transform-{timestamp}"
ROLE_ARN = "arn:aws:iam::993768311527:role/LabRole"
REGION = "us-east-1"

S3_BUCKET = "sagemaker-us-east-1-993768311527"
MODEL_ARTIFACT = f"s3://{S3_BUCKET}/logistic/logistic_model.tar.gz"
INPUT_DATA = f"s3://{S3_BUCKET}/cardio_data/cardio_prod_no_label.csv"
OUTPUT_PATH = f"s3://{S3_BUCKET}/logistic/output/"

# Script Mode Python container (generic Python environment)
PYTHON_IMAGE = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:0.23-1-cpu-py3"

sagemaker = boto3.client("sagemaker")

# Create model with Script Mode environment
sagemaker.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        "Image": PYTHON_IMAGE,
        "ModelDataUrl": MODEL_ARTIFACT,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": MODEL_ARTIFACT,
            "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
            "SAGEMAKER_REGION": REGION
        }
    }
)

# Launch batch transform job
sagemaker.create_transform_job(
    TransformJobName=TRANSFORM_JOB_NAME,
    ModelName=MODEL_NAME,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": INPUT_DATA
            }
        },
        "ContentType": "text/csv",
        "SplitType": "Line"
    },
    TransformOutput={
        "S3OutputPath": OUTPUT_PATH
    },
    TransformResources={
        "InstanceType": "ml.m5.large",
        "InstanceCount": 1
    }
)

print(f"Batch transform started: {TRANSFORM_JOB_NAME}")
print(f"Model: {MODEL_NAME}")

## deploy_model.py
### Deploys the packaged model to an endpoint or prepares it for batch inference on AWS SageMaker.

### Key functions:
  * Uploads the model package to Amazon S3.
  * Registers the model in SageMaker using the Model API.
  * Optionally creates an endpoint for real-time inference or sets up batch transform for offline predictions
  